# SQL Injection Detection Using Machine Learning
## Thesis Presentation — Technical Analysis

---

### Goal
Develop a machine learning-based system for detecting SQL injection attacks in real time.

### Objectives
1. Collect and preprocess a labeled dataset of SQL queries
2. Train and compare multiple machine learning models
3. Select the best-performing model based on key metrics
4. Demonstrate real-time detection with test examples
5. Design the system architecture for production deployment

---

## 1. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import time
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='husl')

RANDOM_STATE = 42
print("All libraries loaded successfully")
print(f"  pandas      {pd.__version__}")
print(f"  numpy       {np.__version__}")
import sklearn; print(f"  scikit-learn {sklearn.__version__}")

---
## 2. Dataset Description and Loading

### Dataset Source
**SQL Injection Dataset** — extended collection of labeled SQL queries  
- **Source**: Kaggle — [SQL Injection Dataset](https://www.kaggle.com/datasets/syedsaqlainhussain/sql-injection-dataset) (extended with synthetic augmentation)
- **Format**: CSV, two columns: `Query` (raw SQL text), `Label` (0 = SAFE, 1 = INJECTION)
- **Size**: 60,381 records

### Attack Types Covered
| Attack Type | Example |
|---|---|
| Boolean-based | `' OR 1=1--` |
| UNION-based | `' UNION SELECT user,pass FROM users--` |
| Time-based | `'; WAITFOR DELAY '0:0:5'--` |
| Stacked queries | `'; DROP TABLE users--` |
| Error-based | `' AND EXTRACTVALUE(1, CONCAT(0x7e, (SELECT version())))` |
| Comment truncation | `admin'--` |

In [ ]:
# Load the dataset
DATASET_PATH = Path("SQL_Dataset_Extended.csv")

df = pd.read_csv(DATASET_PATH)
df.columns = df.columns.str.strip()

# Standardise column names (handle possible variations)
if 'Query' not in df.columns:
    df.rename(columns={df.columns[0]: 'Query', df.columns[1]: 'Label'}, inplace=True)

df = df.dropna(subset=['Query', 'Label'])
df['Query'] = df['Query'].astype(str).str.strip()
df['Label'] = df['Label'].astype(int)

print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)
print(f"Total records    : {len(df):,}")
print(f"SAFE  (Label=0)  : {(df.Label==0).sum():,} ({(df.Label==0).mean()*100:.1f}%)")
print(f"INJECTION (Label=1): {(df.Label==1).sum():,} ({(df.Label==1).mean()*100:.1f}%)")
print(f"Avg query length : {df.Query.str.len().mean():.1f} chars")
print(f"Max query length : {df.Query.str.len().max():,} chars")
print()
print("Sample records:")
df.sample(6, random_state=RANDOM_STATE)[['Query','Label']]

## 3. Data Exploration and Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Dataset Analysis', fontsize=14, fontweight='bold', y=1.02)

# --- Class distribution (pie) ---
counts = df['Label'].value_counts().sort_index()
labels_pie = ['SAFE (0)', 'INJECTION (1)']
colors = ['#2196F3', '#F44336']
axes[0].pie(counts, labels=labels_pie, autopct='%1.1f%%', colors=colors,
            startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Class Distribution', fontweight='bold')

# --- Query length distribution ---
df['query_len'] = df['Query'].str.len()
for label, color, name in [(0,'#2196F3','SAFE'), (1,'#F44336','INJECTION')]:
    subset = df[df.Label == label]['query_len'].clip(upper=300)
    axes[1].hist(subset, bins=40, alpha=0.65, color=color, label=name, density=True)
axes[1].set_xlabel('Query Length (chars)')
axes[1].set_ylabel('Density')
axes[1].set_title('Query Length Distribution', fontweight='bold')
axes[1].legend()

# --- Top special characters in injections ---
injection_queries = df[df.Label == 1]['Query']
special_chars = ["'", '"', '--', '/*', 'OR', 'AND', 'UNION', 'SELECT', 'DROP', 'INSERT']
char_counts = {c: injection_queries.str.upper().str.count(c.upper()).sum() for c in special_chars}
char_df = pd.Series(char_counts).sort_values(ascending=True)
char_df.plot(kind='barh', ax=axes[2], color='#FF7043')
axes[2].set_xlabel('Occurrence Count')
axes[2].set_title('Top SQL Injection Keywords\n(in attack samples)', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_dataset_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print("Figure saved: plot_dataset_analysis.png")

## 4. Data Collection and Analysis Pipeline

```
Raw Dataset (CSV)
      │
      ▼
  Preprocessing
  (clean, strip, normalise)
      │
      ▼
  Feature Extraction
  (TF-IDF vectorisation)
      │
      ▼
  Train / Test Split  ──►  80% Train / 20% Test
      │
      ▼
  Model Training
  (LR · RF · SVM · DT · NB)
      │
      ▼
  Evaluation
  (Accuracy · Precision · Recall · F1 · ROC-AUC)
      │
      ▼
  Best Model → Deployment
```

In [ ]:
# ── Pipeline diagram using matplotlib ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4)
ax.axis('off')
ax.set_title('Data Collection and Analysis Pipeline', fontsize=13, fontweight='bold', pad=10)

steps = [
    ('Raw\nDataset\n(CSV)', '#BBDEFB'),
    ('Pre-\nprocessing', '#C8E6C9'),
    ('Feature\nExtraction\n(TF-IDF)', '#FFF9C4'),
    ('Train/Test\nSplit\n80/20', '#FFE0B2'),
    ('Model\nTraining\n5 models', '#F8BBD9'),
    ('Evaluation\n& Metrics', '#E1BEE7'),
    ('Best Model\nDeployment', '#B2DFDB'),
]

box_w, box_h = 1.6, 2.0
gap = 0.35
start_x = 0.3

for i, (label, color) in enumerate(steps):
    x = start_x + i * (box_w + gap)
    y = 1.0
    rect = mpatches.FancyBboxPatch((x, y), box_w, box_h,
                                    boxstyle="round,pad=0.1",
                                    facecolor=color, edgecolor='#555', linewidth=1.2)
    ax.add_patch(rect)
    ax.text(x + box_w/2, y + box_h/2, label,
            ha='center', va='center', fontsize=8.5, fontweight='bold', color='#222')
    if i < len(steps) - 1:
        arrow_x = x + box_w + 0.02
        ax.annotate('', xy=(arrow_x + gap - 0.04, y + box_h/2),
                    xytext=(arrow_x, y + box_h/2),
                    arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

plt.tight_layout()
plt.savefig('plot_pipeline.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. Feature Extraction — TF-IDF Preprocessing

In [ ]:
# ── TF-IDF Vectorisation ───────────────────────────────────────────────────
X = df['Query']
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

tfidf = TfidfVectorizer(
    analyzer='char_wb',   # character n-grams capture SQL patterns robustly
    ngram_range=(2, 4),
    max_features=30_000,
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("=" * 50)
print("FEATURE EXTRACTION")
print("=" * 50)
print(f"Vectoriser      : TF-IDF (char n-grams 2-4)")
print(f"Vocabulary size : {len(tfidf.vocabulary_):,} features")
print(f"Training samples: {X_train_tfidf.shape[0]:,}")
print(f"Test samples    : {X_test_tfidf.shape[0]:,}")
print(f"Feature matrix  : {X_train_tfidf.shape[0]} × {X_train_tfidf.shape[1]:,}")
print(f"Sparsity        : {(1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0]*X_train_tfidf.shape[1]))*100:.2f}%")

## 6. Mathematical Model — Logistic Regression

### Logistic Regression Formula

$$P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$

Where:
- $\mathbf{x}$ — TF-IDF feature vector of the SQL query  
- $\mathbf{w}$ — learned weight vector  
- $b$ — bias term  
- $\sigma$ — sigmoid activation function  

### Decision Rule
$$\hat{y} = \begin{cases} 1 \ (\text{SQL Injection}) & \text{if } P(y=1 \mid \mathbf{x}) > 0.5 \\ 0 \ (\text{Safe}) & \text{otherwise} \end{cases}$$

### Cost Function (Log-Loss + L2 Regularisation)

$$\mathcal{L}(\mathbf{w}) = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)\right] + \frac{\lambda}{2}\|\mathbf{w}\|^2$$

In [ ]:
# Visualise the sigmoid function
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Sigmoid curve
z = np.linspace(-8, 8, 400)
sigma = 1 / (1 + np.exp(-z))
axes[0].plot(z, sigma, 'b-', lw=2.5, label=r'$\sigma(z) = \frac{1}{1+e^{-z}}$')
axes[0].axhline(0.5, color='r', linestyle='--', lw=1.5, label='Decision boundary (0.5)')
axes[0].axvline(0, color='grey', linestyle=':', lw=1)
axes[0].fill_between(z, 0.5, sigma, where=(sigma > 0.5), alpha=0.15, color='red',
                      label='SQL Injection region')
axes[0].fill_between(z, 0, sigma, where=(sigma <= 0.5), alpha=0.15, color='blue',
                      label='Safe region')
axes[0].set_xlabel(r'$z = \mathbf{w}^T\mathbf{x} + b$', fontsize=12)
axes[0].set_ylabel(r'$P(y=1 \mid \mathbf{x})$', fontsize=12)
axes[0].set_title('Sigmoid Activation — Logistic Regression', fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.4)

# Log-loss visualisation
p = np.linspace(0.001, 0.999, 300)
loss_1 = -np.log(p)           # loss when y=1
loss_0 = -np.log(1 - p)       # loss when y=0
axes[1].plot(p, loss_1, 'r-', lw=2.5, label='y=1 (Injection): $-\\log(p)$')
axes[1].plot(p, loss_0, 'b-', lw=2.5, label='y=0 (Safe): $-\\log(1-p)$')
axes[1].set_xlabel(r'Predicted probability $\hat{p}$', fontsize=12)
axes[1].set_ylabel('Log-Loss', fontsize=12)
axes[1].set_title('Log-Loss (Cross-Entropy)', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 6)
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('plot_sigmoid_logloss.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Training Five Machine Learning Models

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Train a model and return a dict of evaluation metrics."""
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    train_time = time.perf_counter() - t0

    y_pred = model.predict(X_te)

    # Probability scores: use predict_proba when available, else sigmoid(decision_function)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te)[:, 1]
    else:
        raw = model.decision_function(X_te)
        y_prob = 1.0 / (1.0 + np.exp(-raw))   # Platt sigmoid scaling

    return {
        'Model': name,
        'Accuracy':       round(accuracy_score(y_te, y_pred)  * 100, 2),
        'Precision':      round(precision_score(y_te, y_pred) * 100, 2),
        'Recall':         round(recall_score(y_te, y_pred)    * 100, 2),
        'F1-Score':       round(f1_score(y_te, y_pred)        * 100, 2),
        'ROC-AUC':        round(roc_auc_score(y_te, y_prob)   * 100, 2),
        'Train Time (s)': round(train_time, 2),
        '_y_pred': y_pred,
        '_y_prob': y_prob,
        '_model':  model,
    }

# ── Define models ──────────────────────────────────────────────────────────
models_config = [
    ('Logistic Regression', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                                               random_state=RANDOM_STATE)),
    ('Random Forest',       RandomForestClassifier(n_estimators=200, max_depth=None,
                                                   n_jobs=-1, random_state=RANDOM_STATE)),
    ('SVM (Linear)',        LinearSVC(C=0.5, max_iter=2000, random_state=RANDOM_STATE)),
    ('Decision Tree',       DecisionTreeClassifier(max_depth=25, random_state=RANDOM_STATE)),
    ('Naive Bayes',         MultinomialNB(alpha=0.1)),
]

print("Training models — please wait...\n")
results = []
for name, model in models_config:
    print(f"  [{name}] ... ", end='', flush=True)
    result = evaluate_model(name, model, X_train_tfidf, y_train, X_test_tfidf, y_test)
    results.append(result)
    print(f"Accuracy {result['Accuracy']:.2f}%  |  F1 {result['F1-Score']:.2f}%  |  "
          f"ROC-AUC {result['ROC-AUC']:.2f}%  |  {result['Train Time (s)']}s")

print("\nAll models trained.")

## 8. Model Comparison — Results Table

In [ ]:
metric_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Train Time (s)']
results_df = pd.DataFrame(results)[metric_cols].sort_values('F1-Score', ascending=False)
results_df = results_df.reset_index(drop=True)
results_df.index += 1  # start rank at 1

print("=" * 80)
print("MODEL COMPARISON — ALL METRICS (%)")
print("=" * 80)
print(results_df.to_string())
print("=" * 80)

best_model_name = results_df.iloc[0]['Model']
best_acc  = results_df.iloc[0]['Accuracy']
best_f1   = results_df.iloc[0]['F1-Score']
best_auc  = results_df.iloc[0]['ROC-AUC']
print(f"\n★  Best model : {best_model_name}")
print(f"   Accuracy   : {best_acc:.2f}%")
print(f"   F1-Score   : {best_f1:.2f}%")
print(f"   ROC-AUC    : {best_auc:.2f}%")

## 9. Performance Graphs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

metric_plot_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_names = results_df['Model'].tolist()
x = np.arange(len(model_names))
width = 0.15
colors_bar = ['#1976D2', '#388E3C', '#F57C00', '#7B1FA2', '#C62828']

# Grouped bar chart
for i, (metric, color) in enumerate(zip(metric_plot_cols, colors_bar)):
    vals = results_df[metric].values
    bars = axes[0].bar(x + i * width, vals, width, label=metric, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                     f'{val:.1f}', ha='center', va='bottom', fontsize=6.5, rotation=45)

axes[0].set_xticks(x + width * 2)
axes[0].set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('Score (%)')
axes[0].set_ylim(50, 105)
axes[0].set_title('All Metrics by Model', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=8)
axes[0].grid(axis='y', alpha=0.4)

# F1-Score bar chart (sorted)
f1_vals = results_df.set_index('Model')['F1-Score'].sort_values()
bar_colors = ['#EF5350' if m == best_model_name else '#78909C' for m in f1_vals.index]
bars = axes[1].barh(f1_vals.index, f1_vals.values, color=bar_colors, edgecolor='#333', linewidth=0.7)
for bar, val in zip(bars, f1_vals.values):
    axes[1].text(val + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}%', va='center', fontsize=10, fontweight='bold')
axes[1].set_xlim(50, 105)
axes[1].set_xlabel('F1-Score (%)')
axes[1].set_title('F1-Score Ranking', fontweight='bold')
axes[1].grid(axis='x', alpha=0.4)
red_patch = mpatches.Patch(color='#EF5350', label='Best model')
grey_patch = mpatches.Patch(color='#78909C', label='Other models')
axes[1].legend(handles=[red_patch, grey_patch], fontsize=9)

plt.tight_layout()
plt.savefig('plot_model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## 10. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
roc_colors = ['#1976D2', '#E53935', '#43A047', '#FB8C00', '#8E24AA']
linestyles = ['-', '-', '-', '--', '-.']

for res, color, ls in zip(results, roc_colors, linestyles):
    fpr, tpr, _ = roc_curve(y_test, res['_y_prob'])
    auc_val = res['ROC-AUC']
    ax.plot(fpr, tpr, color=color, lw=2.2, ls=ls,
            label=f"{res['Model']} (AUC = {auc_val:.2f}%)")

ax.plot([0,1], [0,1], 'k--', lw=1.2, label='Random Classifier (AUC = 50%)')
ax.fill_between([0,1], [0,0], [1,1], alpha=0.03, color='grey')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('plot_roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()

## 11. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(results), figsize=(20, 4))
fig.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold')

for ax, res in zip(axes, results):
    cm = confusion_matrix(y_test, res['_y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Safe', 'Injection'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f"{res['Model']}\nAcc={res['Accuracy']:.1f}%", fontsize=9, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=8)
    ax.set_ylabel('Actual', fontsize=8)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.savefig('plot_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()

## 12. Best Model — Detailed Classification Report

In [ ]:
# Find the best result dict
best_result = max(results, key=lambda r: r['F1-Score'])
print(f"=" * 60)
print(f"DETAILED REPORT — {best_result['Model'].upper()}")
print(f"=" * 60)
print(classification_report(y_test, best_result['_y_pred'],
                             target_names=['SAFE (0)', 'INJECTION (1)'],
                             digits=4))
print(f"ROC-AUC Score : {best_result['ROC-AUC']:.4f}%")

# Summary table
print("\n" + "=" * 60)
print(f"METRIC SUMMARY — {best_result['Model'].upper()}")
print("=" * 60)
for metric in ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']:
    bar_len = int(best_result[metric] / 2)
    bar = '█' * bar_len + '░' * (50 - bar_len)
    print(f"  {metric:<12} {best_result[metric]:6.2f}%  |{bar}|")

## 13. System Architecture

### Components
| Layer | Component | Technology |
|-------|-----------|------------|
| Client | Browser / API consumer | HTTP / HTTPS |
| Web Application | REST API server | FastAPI + Uvicorn |
| Detection Engine | Feature extraction | TF-IDF |
| ML Model | Classification | Scikit-learn (best model) |
| Logging | Incident persistence | SQLite / Redis |
| Monitoring | Metrics export | Prometheus |

### Technology Stack
| Category | Technology | Version |
|---|---|---|
| Language | Python | 3.10+ |
| API Framework | FastAPI | 0.136 |
| ML Library | Scikit-learn | 1.8 |
| Data Processing | Pandas, NumPy | 3.0 / 2.4 |
| Deep Learning | PyTorch | 2.12 |
| Containerisation | Docker | 24+ |
| Monitoring | Prometheus | 0.25 |

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('SQL Injection Detection System — Architecture', fontsize=14, fontweight='bold', y=0.98)

def draw_box(ax, x, y, w, h, label, sublabel='', color='#E3F2FD', fontsize=9):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12",
                                    facecolor=color, edgecolor='#37474F', linewidth=1.5, zorder=3)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.18 if sublabel else 0), label,
            ha='center', va='center', fontsize=fontsize, fontweight='bold', color='#1A237E', zorder=4)
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.22, sublabel,
                ha='center', va='center', fontsize=7.5, color='#455A64', zorder=4, style='italic')

def draw_arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#546E7A', lw=1.8), zorder=2)
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx + 0.1, my, label, fontsize=7.5, color='#546E7A', style='italic')

# ── Client ──
draw_box(ax, 0.4, 8.0, 3.0, 1.2, 'CLIENT', 'Browser / API Consumer', '#BBDEFB', fontsize=10)

# ── FastAPI Server ──
draw_box(ax, 5.0, 8.0, 4.0, 1.2, 'FastAPI Server', 'REST API / Uvicorn', '#C8E6C9', fontsize=10)

# ── Feature Extractor ──
draw_box(ax, 3.5, 5.8, 3.0, 1.2, 'Feature Extractor', 'TF-IDF (char n-grams)', '#FFF9C4', fontsize=9)

# ── ML Model ──
draw_box(ax, 7.0, 5.8, 3.0, 1.2, 'ML Model', 'Scikit-learn Classifier', '#FFE0B2', fontsize=9)

# ── Decision Engine ──
draw_box(ax, 5.0, 3.8, 4.0, 1.2, 'Decision Engine', 'Threshold + Rules', '#F8BBD9', fontsize=9)

# ── Incident Logger ──
draw_box(ax, 0.4, 3.8, 3.0, 1.2, 'Incident Logger', 'SQLite / Redis', '#E1BEE7', fontsize=9)

# ── Prometheus ──
draw_box(ax, 10.5, 3.8, 3.0, 1.2, 'Monitoring', 'Prometheus / Grafana', '#B2DFDB', fontsize=9)

# ── Detection Result ──
draw_box(ax, 4.5, 1.5, 5.0, 1.2, 'Detection Result', 'SAFE / INJECTION + Severity', '#FFCDD2', fontsize=10)

# ── Arrows ──
draw_arrow(ax, 3.4,  8.6,  5.0,  8.6,  'HTTP POST /api/check')
draw_arrow(ax, 7.0,  8.0,  6.5,  7.0)          # server → feature extractor
draw_arrow(ax, 5.0,  6.4,  7.0,  6.4)          # feature extractor → ML model
draw_arrow(ax, 8.5,  5.8,  8.0,  5.0)          # ML model → decision engine
draw_arrow(ax, 5.0,  4.4,  3.4,  4.4)          # decision engine → logger
draw_arrow(ax, 9.0,  4.4, 10.5,  4.4)          # decision engine → monitoring
draw_arrow(ax, 7.0,  3.8,  7.0,  2.7)          # decision engine → result
draw_arrow(ax, 7.0,  1.5,  3.4,  8.2)          # result → client (response)

# Legend
legend_items = [
    mpatches.Patch(facecolor='#BBDEFB', edgecolor='#37474F', label='Client Layer'),
    mpatches.Patch(facecolor='#C8E6C9', edgecolor='#37474F', label='API Layer'),
    mpatches.Patch(facecolor='#FFF9C4', edgecolor='#37474F', label='Feature Layer'),
    mpatches.Patch(facecolor='#FFE0B2', edgecolor='#37474F', label='ML Layer'),
    mpatches.Patch(facecolor='#F8BBD9', edgecolor='#37474F', label='Decision Layer'),
    mpatches.Patch(facecolor='#FFCDD2', edgecolor='#37474F', label='Output Layer'),
]
ax.legend(handles=legend_items, loc='lower right', fontsize=8, ncol=2,
          framealpha=0.9, title='Architecture Layers', title_fontsize=8)

plt.tight_layout()
plt.savefig('plot_architecture.png', bbox_inches='tight', dpi=150)
plt.show()

## 14. SQL Injection Testing — Live Demo

In [ ]:
# Build a simple detection function using the best trained model
best_model_obj = max(results, key=lambda r: r['F1-Score'])['_model']

def detect_sql_injection(query: str) -> dict:
    """Run TF-IDF + best model classification on a raw SQL query."""
    vec = tfidf.transform([query])
    pred = best_model_obj.predict(vec)[0]
    if hasattr(best_model_obj, 'predict_proba'):
        prob = best_model_obj.predict_proba(vec)[0][1]
    elif hasattr(best_model_obj, 'decision_function'):
        raw = best_model_obj.decision_function(vec)[0]
        prob = 1 / (1 + np.exp(-raw))  # sigmoid
    else:
        prob = float(pred)

    label = 'SQL INJECTION' if pred == 1 else 'SAFE'
    action = 'BLOCK' if pred == 1 else 'ALLOW'
    severity = 'HIGH' if prob > 0.85 else ('MEDIUM' if prob > 0.60 else ('LOW' if pred == 1 else 'NONE'))
    return {
        'query': query,
        'label': label,
        'probability': round(prob * 100, 2),
        'action': action,
        'severity': severity,
    }

# ── Test queries ────────────────────────────────────────────────────────────
test_cases = [
    # Injection examples
    ("' OR 1=1--",                                          "Boolean-based"),
    ("admin' OR 'a'='a",                                    "Boolean-based"),
    ("1; DROP TABLE users--",                               "Stacked query"),
    ("' UNION SELECT username, password FROM users--",      "UNION-based"),
    ("; WAITFOR DELAY '0:0:5'--",                           "Time-based"),
    ("1' AND EXTRACTVALUE(1, CONCAT(0x7e,(SELECT version())))", "Error-based"),
    # Safe examples
    ("SELECT * FROM products WHERE category='Books'",       "Safe query"),
    ("UPDATE users SET email='john@example.com' WHERE id=5", "Safe update"),
    ("SELECT name FROM customers WHERE city = 'O\'Brien\'s'", "Safe (apostrophe in name)"),
    ("2024-01-15",                                          "Safe date input"),
]

print(f"{'─'*95}")
print(f"  {'INPUT QUERY':<45} {'TYPE':<22} {'RESULT':<18} {'PROB':>6}  {'ACTION':<8} {'SEV':<8}")
print(f"{'─'*95}")

for query, qtype in test_cases:
    res = detect_sql_injection(query)
    icon = '🔴' if res['label'] == 'SQL INJECTION' else '🟢'
    short_q = (query[:42] + '...') if len(query) > 45 else query
    print(f"  {short_q:<45} {qtype:<22} {icon} {res['label']:<15} {res['probability']:>6.1f}%  {res['action']:<8} {res['severity']:<8}")

print(f"{'─'*95}")

In [ ]:
# Visual test results chart
fig, ax = plt.subplots(figsize=(13, 6))

test_results = [detect_sql_injection(q) for q, _ in test_cases]
test_labels  = [t[1] for t in test_cases]
probs  = [r['probability'] for r in test_results]
labels = [r['label'] for r in test_results]
bar_colors = ['#E53935' if l == 'SQL INJECTION' else '#43A047' for l in labels]

bars = ax.barh(range(len(test_cases)), probs, color=bar_colors, edgecolor='#333', linewidth=0.7, height=0.65)
ax.axvline(50, color='orange', linestyle='--', lw=1.8, label='Decision threshold (50%)')

for i, (bar, label, tl) in enumerate(zip(bars, labels, test_labels)):
    prob_val = bar.get_width()
    ax.text(min(prob_val + 1, 101), bar.get_y() + bar.get_height()/2,
            f'{prob_val:.1f}%  [{label}]', va='center', fontsize=8.5)

y_labels = [f"{t[1]}\n{t[0][:38]}{'...' if len(t[0])>38 else ''}" for t in test_cases]
ax.set_yticks(range(len(test_cases)))
ax.set_yticklabels(y_labels, fontsize=8)
ax.set_xlabel('Injection Probability (%)', fontsize=11)
ax.set_title('SQL Injection Testing — Live Detection Results', fontsize=13, fontweight='bold')
ax.set_xlim(0, 115)
ax.grid(axis='x', alpha=0.4)

red_patch  = mpatches.Patch(color='#E53935', label='SQL INJECTION — BLOCK')
green_patch = mpatches.Patch(color='#43A047', label='SAFE — ALLOW')
ax.legend(handles=[red_patch, green_patch, ax.get_lines()[0]], fontsize=9, loc='lower right')

plt.tight_layout()
plt.savefig('plot_detection_demo.png', bbox_inches='tight', dpi=150)
plt.show()

## 15. Conclusion

### Objectives Completion

In [ ]:
objectives = [
    ('Dataset collected and preprocessed',      '✓', f"{len(df):,} queries, 2 classes"),
    ('Multiple ML models trained',              '✓', 'LR, RF, SVM, DT, Naive Bayes'),
    ('Performance metrics evaluated',           '✓', 'Accuracy, Precision, Recall, F1, ROC-AUC'),
    ('Best model selected',                     '✓', f"{best_result['Model']} — F1={best_result['F1-Score']:.2f}%"),
    ('Real-time detection demonstrated',        '✓', '10 test queries (6 attacks, 4 safe)'),
    ('System architecture designed',            '✓', '6-layer FastAPI + ML pipeline'),
    ('Goal achieved',                           '✓', f"Detection accuracy {best_result['Accuracy']:.2f}%"),
]

print("=" * 75)
print("CONCLUSION — OBJECTIVES AND STATUS")
print("=" * 75)
for obj, status, detail in objectives:
    print(f"  {status}  {obj:<45}  {detail}")
print("=" * 75)

print(f"""
★  FINAL SUMMARY

   The developed system successfully detects SQL injection attacks
   in real time with an accuracy of {best_result['Accuracy']:.2f}% and F1-Score of
   {best_result['F1-Score']:.2f}%, demonstrating its effectiveness for enhancing
   web application security.

   Best model   : {best_result['Model']}
   Accuracy     : {best_result['Accuracy']:.2f}%
   F1-Score     : {best_result['F1-Score']:.2f}%
   ROC-AUC      : {best_result['ROC-AUC']:.2f}%
   Dataset size : {len(df):,} queries
""")

In [ ]:
# Final summary comparison table visualised
fig, ax = plt.subplots(figsize=(12, 4))
ax.axis('off')

table_data = [
    [r['Model'], f"{r['Accuracy']:.2f}%", f"{r['Precision']:.2f}%",
     f"{r['Recall']:.2f}%", f"{r['F1-Score']:.2f}%", f"{r['ROC-AUC']:.2f}%"]
    for r in sorted(results, key=lambda x: x['F1-Score'], reverse=True)
]

col_headers = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
tbl = ax.table(cellText=table_data, colLabels=col_headers,
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.0, 2.2)

# Header styling
for j in range(len(col_headers)):
    tbl[0, j].set_facecolor('#1565C0')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Best model row styling
for j in range(len(col_headers)):
    tbl[1, j].set_facecolor('#E8F5E9')
    tbl[1, j].set_text_props(fontweight='bold', color='#1B5E20')

# Alternate row shading
for i in range(2, len(table_data) + 1):
    for j in range(len(col_headers)):
        tbl[i, j].set_facecolor('#F5F5F5' if i % 2 == 0 else 'white')

ax.set_title('Model Comparison — Final Summary Table (ranked by F1-Score)',
             fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('plot_final_table.png', bbox_inches='tight', dpi=150)
plt.show()

---
## References

1. **Dataset**: Syyed Saqlainhussain. *SQL Injection Dataset*. Kaggle, 2022.  
   https://www.kaggle.com/datasets/syedsaqlainhussain/sql-injection-dataset

2. **Logistic Regression**: Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.

3. **Random Forest**: Breiman, L. (2001). Random Forests. *Machine Learning, 45*(1), 5–32.

4. **TF-IDF Feature Extraction**: Salton, G., & Buckley, C. (1988). Term-weighting approaches in automatic text retrieval. *Information Processing & Management, 24*(5), 513–523.

5. **SQL Injection (OWASP)**: OWASP Foundation. *OWASP Top 10: Injection*. https://owasp.org/Top10/A03_2021-Injection/

6. **Scikit-learn**: Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR, 12*, 2825–2830.

7. **FastAPI**: Ramírez, S. (2019). *FastAPI Framework*. https://fastapi.tiangolo.com/

8. **PyTorch**: Paszke, A., et al. (2019). PyTorch: An Imperative Style, High-Performance Deep Learning Library. *NeurIPS*.

---
*Notebook generated for thesis defence presentation.*  
*Dataset: SQL_Dataset_Extended.csv (60,381 records) | Framework: scikit-learn 1.8 | Python 3.10+*